# Проверка утечки через имена говорящих

**Вопрос.** Базовая модель `xlm-roberta-base` воспроизвела silver-метки на всех 403
диалогах теста (100%), а val F1 достиг 1.0 уже к третьей эпохе. Гипотеза: задача
решается не по содержанию диалога, а по имени говорящего, которое попадает в текст
как `[2|Security Service] …`.

Основания для гипотезы (посчитано на корпусе): **92.1%** имён atk-говорящих
(488 из 530) встречаются только в scam либо только в benign; **87.2%** привязаны
ровно к одному сценарию. При этом сама роль меткой НЕ является — `atk` есть во всех
2000 диалогов, и в 999 benign, и в 1001 scam, так что заменять имя на роль корректно.

**Эксперимент.** Два прогона, идентичные во всём (сид, гиперпараметры, сплиты,
порядок данных), кроме текста:

| прогон | текст реплики |
|---|---|
| `names` (контроль) | `[2\|Security Service] We detected a login…` |
| `noNames` (проверка) | `[2\|atk] We detected a login…` |

**Как читать результат.** Если без имён val F1 заметно падает и модель перестаёт
воспроизводить silver — виновата утечка, базовые числа непригодны. Если качество
держится — дело не в именах, а в стилевых артефактах генератора, и это вопрос к
корпусу, а не к модели.

**Настройки Kaggle:** Accelerator = GPU T4, Internet = On. Датасет с
`corpus_v1.jsonl`, `gold_test_v1.csv`, `gold_dev_split.json`. Время — примерно
2 × 8 минут.

## Конфигурация и данные

In [ ]:
import os, sys, json, csv, glob, pathlib, time, copy

def find_one(name, required=True):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits and not required:
        print(f"(необязательный {name} не найден)")
        return None
    assert hits, f"не найден {name} под /kaggle/input — подключён ли датасет?"
    return hits[0]

CORPUS   = find_one("corpus_v1.jsonl")
GOLD     = find_one("gold_test_v1.csv")
GOLD_DEV = find_one("gold_dev_split.json", required=False)

OUT = pathlib.Path("/kaggle/working/leak_check"); OUT.mkdir(parents=True, exist_ok=True)
TB_ROOT = pathlib.Path("/kaggle/working/tb")

MODEL_NAME, SEED = "xlm-roberta-base", 20260721
LR, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.2
EPOCHS, MAX_LENGTH, BATCH_SIZE = 5, 512, 16

import torch, transformers
DEVICE, USE_FP16 = ("cuda", True) if torch.cuda.is_available() else ("cpu", False)
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("устройство:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
def read_corpus(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line: out.append(json.loads(line))
    return out

def build_text(rec, anonymize_speakers=False):
    """anonymize_speakers=True → вместо имени подставляется роль atk/tgt."""
    key = "role" if anonymize_speakers else "speaker"
    return "\n".join(f"[{t['i']}|{t[key]}] {t['text']}" for t in rec["turns"])

def read_gold_csv(path):
    out = {}
    with open(path, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f):
            did = (r.get("dialog_id") or "").strip()
            if did: out[did] = int(r["is_scam"])
    return out

records = read_corpus(CORPUS)
gold    = read_gold_csv(GOLD)

train_recs = [r for r in records if r.get("split") == "train"]
val_recs   = [r for r in records if r.get("split") == "val"]

test_split = [r for r in records if r.get("split") == "test"]
test_recs = []
for r in test_split:
    r2 = {k: v for k, v in r.items() if k != "is_scam"}   # silver-метку выбрасываем
    if r2["id"] in gold:
        r2["gold_is_scam"] = gold[r2["id"]]
        test_recs.append(r2)
assert len(test_recs) == len(gold), f"джойн неполный: {len(test_recs)} из {len(gold)}"
print(f"джойн gold ↔ test: {len(test_recs)}/{len(gold)}")

dev_recs = []
if GOLD_DEV:
    assign = json.load(open(GOLD_DEV, encoding="utf-8"))["assignment"]
    dev_recs  = [r for r in test_recs if assign.get(r["id"]) == "dev"]
    test_recs = [r for r in test_recs if assign.get(r["id"]) != "dev"]
print(f"train {len(train_recs)} / val {len(val_recs)} / gold-dev {len(dev_recs)} / test {len(test_recs)}")

# контроль подстановки
_r = train_recs[0]
print("\nС ИМЕНАМИ: ", build_text(_r).split("\n")[0][:88])
print("БЕЗ ИМЁН:  ", build_text(_r, True).split("\n")[0][:88])

## Один прогон как функция

Оба запуска идут через неё, поэтому отличаться могут только флагом `anonymize`.

In [ ]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)
from sklearn.metrics import (accuracy_score, cohen_kappa_score, roc_auc_score,
                             precision_recall_fscore_support)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def metrics_from(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary",
                                                 pos_label=1, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)), "precision": float(p),
            "recall": float(r), "f1": float(f),
            "cohen_kappa": float(cohen_kappa_score(y_true, y_pred))}

class DlgDataset(torch.utils.data.Dataset):
    def __init__(self, recs, label_key, anonymize):
        self.enc = tok([build_text(r, anonymize) for r in recs], truncation=True,
                       max_length=MAX_LENGTH, padding="max_length")
        self.labels = [int(r[label_key]) for r in recs]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i]); return item

def run_experiment(anonymize, tag):
    set_seed(SEED)                       # одинаковая инициализация в обоих прогонах
    t0 = time.time()
    ds_train = DlgDataset(train_recs, "is_scam", anonymize)
    ds_val   = DlgDataset(val_recs,   "is_scam", anonymize)
    ds_test  = DlgDataset(test_recs,  "gold_is_scam", anonymize)
    ds_dev   = DlgDataset(dev_recs,   "gold_is_scam", anonymize) if dev_recs else None

    if ds_dev is not None:
        eval_sets, best_metric = {"silver_val": ds_val, "gold_dev": ds_dev}, "eval_gold_dev_f1"
    else:
        eval_sets, best_metric = ds_val, "f1"

    tb_dir = TB_ROOT / f"{MODEL_NAME.split('/')[-1]}_seed{SEED}_e{EPOCHS}_{tag}"
    tb_dir.mkdir(parents=True, exist_ok=True)
    os.environ["TENSORBOARD_LOGGING_DIR"] = str(tb_dir)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    targs = TrainingArguments(
        output_dir=str(OUT / f"ckpt_{tag}"),
        learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        seed=SEED, data_seed=SEED, fp16=USE_FP16,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model=best_metric,
        greater_is_better=True, save_total_limit=1, logging_steps=10,
        report_to="tensorboard", logging_dir=str(tb_dir),
    )
    def compute_metrics(ep):
        logits, labels = ep
        return metrics_from(labels, np.argmax(logits, axis=-1))

    trainer = Trainer(model=model, args=targs, train_dataset=ds_train,
                      eval_dataset=eval_sets, processing_class=tok,
                      compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()

    po = trainer.predict(ds_test)
    logits = po.predictions
    probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    y_pred = logits.argmax(axis=-1)
    y_true = np.array([r["gold_is_scam"] for r in test_recs])

    m = metrics_from(y_true, y_pred)
    m["roc_auc"] = float(roc_auc_score(y_true, probs)) if len(set(y_true.tolist())) > 1 else float("nan")

    # ключевой показатель: насколько модель повторяет ЧЕРНОВУЮ разметку LLM
    silver = {r["id"]: r for r in records}
    same_silver = sum(1 for r, p in zip(test_recs, y_pred) if int(silver[r["id"]]["is_scam"]) == int(p))

    per_lang = {}
    for lg in sorted({r.get("lang") for r in test_recs if r.get("lang")}):
        idx = [i for i, r in enumerate(test_recs) if r.get("lang") == lg]
        per_lang[lg] = metrics_from(y_true[idx], y_pred[idx]) | {"n": len(idx)}

    hist = copy.deepcopy(trainer.state.log_history)
    pfx = "eval_gold_dev_" if dev_recs else "eval_"
    ev = [h for h in hist if f"{pfx}loss" in h]
    sv = [h for h in hist if "eval_silver_val_loss" in h]

    res = {"tag": tag, "anonymize": anonymize, "test_metrics": m, "per_language": per_lang,
           "silver_agreement": same_silver / len(test_recs), "n_test": len(test_recs),
           "best_checkpoint": trainer.state.best_model_checkpoint,
           "best_metric": trainer.state.best_metric, "selection_metric": best_metric,
           "minutes": round((time.time() - t0) / 60, 1),
           "epochs_ran": len(ev), "history": hist,
           "val_silver_f1": [h.get("eval_silver_val_f1") for h in sv],
           "gold_dev_f1": [h.get("eval_gold_dev_f1") for h in ev] if dev_recs else [],
           "probs": probs.tolist(), "y_true": y_true.tolist(), "y_pred": y_pred.tolist()}

    from torch.utils.tensorboard import SummaryWriter
    w = SummaryWriter(log_dir=str(tb_dir))
    for k, v in m.items(): w.add_scalar(f"gold_test/{k}", v, 0)
    for lg, mm in per_lang.items(): w.add_scalar(f"gold_test_f1/{lg}", mm["f1"], 0)
    w.add_scalar("gold_test/silver_agreement", res["silver_agreement"], 0)
    w.flush(); w.close()

    del trainer, model
    torch.cuda.empty_cache()
    print(f"\n[{tag}] готово за {res['minutes']} мин | gold F1 {m['f1']:.4f} | "
          f"совпадение с silver {res['silver_agreement']:.1%}")
    return res

## Прогон 1 — контроль, с именами говорящих

In [ ]:
res_names = run_experiment(anonymize=False, tag="names")

## Прогон 2 — проверка, имена заменены на роли

In [ ]:
res_nonames = run_experiment(anonymize=True, tag="noNames")

## Сравнение и вывод

In [ ]:
A, B = res_names, res_nonames

print("=" * 74)
print(f"{'показатель':<34}{'с именами':>13}{'без имён':>13}{'разница':>13}")
print("=" * 74)
rows = [("gold F1", A["test_metrics"]["f1"], B["test_metrics"]["f1"]),
        ("gold accuracy", A["test_metrics"]["accuracy"], B["test_metrics"]["accuracy"]),
        ("gold precision", A["test_metrics"]["precision"], B["test_metrics"]["precision"]),
        ("gold recall", A["test_metrics"]["recall"], B["test_metrics"]["recall"]),
        ("gold Cohen k", A["test_metrics"]["cohen_kappa"], B["test_metrics"]["cohen_kappa"]),
        ("gold ROC AUC", A["test_metrics"]["roc_auc"], B["test_metrics"]["roc_auc"]),
        ("совпадение с silver", A["silver_agreement"], B["silver_agreement"])]
for name, a, b in rows:
    print(f"{name:<34}{a:>13.4f}{b:>13.4f}{b-a:>+13.4f}")

va, vb = [x for x in A["val_silver_f1"] if x is not None], [x for x in B["val_silver_f1"] if x is not None]
if va and vb:
    print(f"\n{'val F1 (silver) по эпохам':<34}")
    print(f"{'  с именами':<34}{'  '.join(f'{v:.4f}' for v in va)}")
    print(f"{'  без имён':<34}{'  '.join(f'{v:.4f}' for v in vb)}")
    print(f"\nмаксимум val F1: с именами {max(va):.4f}, без имён {max(vb):.4f}")

print("\nпо языкам (gold F1):")
for lg in sorted(A["per_language"]):
    a, b = A["per_language"][lg]["f1"], B["per_language"][lg]["f1"]
    print(f"  {lg}: {a:.4f} → {b:.4f}  ({b-a:+.4f}), n={A['per_language'][lg]['n']}")

d_f1 = B["test_metrics"]["f1"] - A["test_metrics"]["f1"]
d_ag = B["silver_agreement"] - A["silver_agreement"]
print("\n" + "=" * 74)
print("ВЫВОД:")
if d_f1 <= -0.05 or d_ag <= -0.05:
    print("  Убрав имена, качество заметно просело → ИМЕНА ГОВОРЯЩИХ БЫЛИ УТЕЧКОЙ.")
    print("  Базовые числа (gold F1 0.8753) непригодны, нужен пересчёт на обезличенном тексте.")
elif abs(d_f1) < 0.02 and abs(d_ag) < 0.02:
    print("  Разницы практически нет → дело НЕ в именах.")
    print("  Причину искать в стилевых артефактах генератора; это вопрос к корпусу, не к модели.")
else:
    print(f"  Промежуточный результат: F1 {d_f1:+.4f}, совпадение с silver {d_ag:+.4f}.")
    print("  Имена дают часть сигнала, но не весь — смотреть оба фактора.")
print("=" * 74)

json.dump({k: {kk: vv for kk, vv in r.items() if kk not in ("history", "probs")}
           for k, r in (("names", A), ("noNames", B))},
          open(OUT / "leak_check_results.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
json.dump({"names": A["history"], "noNames": B["history"]},
          open(OUT / "leak_check_history.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\nсохранено:", OUT / "leak_check_results.json")

## Графики

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")
PLOTS = OUT / "plots"; PLOTS.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
a0, a1 = axes
for res, lab, c in ((A, "с именами", "#4C72B0"), (B, "без имён", "#C44E52")):
    v = [x for x in res["val_silver_f1"] if x is not None]
    if v: a0.plot(range(1, len(v) + 1), v, marker="o", color=c, linewidth=2.2, label=lab)
a0.set_ylim(0, 1.02); a0.set_xlabel("эпоха"); a0.set_ylabel("val F1 (silver-метки)")
a0.set_title("Валидация на silver: держится ли F1 без имён"); a0.legend(loc="lower right")

keys = ["f1", "accuracy", "precision", "recall", "cohen_kappa"]
x = np.arange(len(keys)); w_ = 0.36
a1.bar(x - w_/2, [A["test_metrics"][k] for k in keys], w_, color="#4C72B0", label="с именами")
a1.bar(x + w_/2, [B["test_metrics"][k] for k in keys], w_, color="#C44E52", label="без имён")
a1.set_xticks(x, ["F1", "acc", "prec", "recall", "kappa"]); a1.set_ylim(0, 1.05)
a1.set_ylabel("значение"); a1.set_title(f"Золотой тест (n={A['n_test']})"); a1.legend(loc="lower right")
fig.suptitle("Проверка утечки через имена говорящих")
fig.tight_layout(); fig.savefig(PLOTS / "leak_check.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(["с именами", "без имён"], [A["silver_agreement"], B["silver_agreement"]],
       color=["#4C72B0", "#C44E52"], width=0.5)
for i, v in enumerate([A["silver_agreement"], B["silver_agreement"]]):
    ax.text(i, v + 0.015, f"{v:.1%}", ha="center", fontsize=13)
ax.set_ylim(0, 1.12); ax.set_ylabel("доля совпадений с silver-меткой LLM")
ax.set_title("Насколько модель воспроизводит черновую разметку\n(100% = выучила разметчика, а не задачу)")
fig.tight_layout(); fig.savefig(PLOTS / "silver_agreement.png", dpi=150); plt.show()
print("графики:", list(PLOTS.glob("*.png")))